# Reranker: candidate union User Tower + Popularity + Content

Notebook này **đóng băng** User Tower checkpoint. Residual listwise reranker học trên 80% validation users, chọn epoch và blend trên 20% validation users, và chỉ báo cáo test một lần.

In [ ]:
import os, json, math, random, hashlib
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 20260813
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.environ.get('DATN_DATA_DIR'):
    DATA_DIR = Path(os.environ['DATN_DATA_DIR'])
elif Path('data/processed/balanced_u5_i2_v1').exists():
    DATA_DIR = Path('data/processed/balanced_u5_i2_v1')
elif Path('/kaggle/input/datasets/hoho0111/clothing-shoes-and-jewelry').exists():
    DATA_DIR = Path('/kaggle/input/datasets/hoho0111/clothing-shoes-and-jewelry')
elif Path('/kaggle/input/clothing-shoes-and-jewelry').exists():
    DATA_DIR = Path('/kaggle/input/clothing-shoes-and-jewelry')
else:
    DATA_DIR = Path('data')

# If reranking in a fresh Kaggle notebook, publish the saved User Tower artifact as a Kaggle Dataset
# and replace this path with its mounted directory.
USER_TOWER_DIR = Path('/kaggle/working/artifacts/user_tower_balanced_v1')
if not USER_TOWER_DIR.exists():
    USER_TOWER_DIR = Path('data/artifacts/user_tower_balanced_v1')
if not (USER_TOWER_DIR / 'user_tower.pt').exists():
    raise FileNotFoundError(f'Không tìm thấy user_tower.pt tại {USER_TOWER_DIR}. Hãy mount Kaggle Dataset checkpoint và cập nhật USER_TOWER_DIR.')

ARTIFACTS_DIR = Path('/kaggle/working/artifacts/reranker_v2') if Path('/kaggle/working').exists() else Path('data/artifacts/reranker_v2')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
ITEMS_PATH, TRAIN_PATH = DATA_DIR/'items.parquet', DATA_DIR/'train.parquet'
VALID_PATH, TEST_PATH = DATA_DIR/'valid.parquet', DATA_DIR/'test.parquet'
EMB_DIR = Path(os.environ['DATN_EMB_DIR']) if os.environ.get('DATN_EMB_DIR') else (Path('data/embedding') if Path('data/embedding').exists() else (DATA_DIR/'embedding' if (DATA_DIR/'embedding').exists() else DATA_DIR))

# Candidate budget. This is the retrieval ceiling; reranker cannot recover a missing target.
# Validation-locked practical recipe: at most 2,000 candidates before de-duplication.
UT_CANDIDATES, POP_CANDIDATES = 1000, 300
CONTENT_CANDIDATES, LAST_ITEM_CANDIDATES = 400, 300
LISTWISE_NEGATIVES = 128
HARD_NEGATIVE_FRACTION = 0.75
RERANK_BATCH_SIZE, EPOCHS, LR = 256, 20, 3e-4
PATIENCE, HR100_TOLERANCE = 4, 5e-4
BLEND_GRID = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0)
MAX_CANDIDATES = UT_CANDIDATES + POP_CANDIDATES + CONTENT_CANDIDATES + LAST_ITEM_CANDIDATES
print(f'Device={device}; retrieval candidates <= {MAX_CANDIDATES}')

In [ ]:
PAD_IDX = 0

@dataclass(frozen=True)
class ItemVocab:
    item2idx: dict
    num_items: int
    @property
    def oov_idx(self): return self.num_items + 1
    @property
    def vocab_size(self): return self.num_items + 2

def pad_right(seq, max_len=20):
    out = np.zeros(max_len, dtype=np.int64); seq = seq[-max_len:]
    out[:len(seq)] = seq
    return out

def load_sequences():
    ids = pl.read_parquet(ITEMS_PATH, columns=['item_id'])['item_id'].to_list()
    vocab = ItemVocab({item_id: i + 1 for i, item_id in enumerate(ids)}, len(ids))
    lookup = pl.DataFrame({'item_id': list(vocab.item2idx), 'item_idx': list(vocab.item2idx.values())})
    def read(path):
        return (pl.read_parquet(path).filter(pl.col('is_positive') == 1).sort('timestamp')
          .join(lookup, on='item_id', how='left').with_columns(pl.col('item_idx').fill_null(vocab.oov_idx))
          .select('user_id', 'item_idx'))
    tr, va, te = read(TRAIN_PATH), read(VALID_PATH), read(TEST_PATH)
    train = {r['user_id']: r['item_idx'] for r in tr.group_by('user_id', maintain_order=True).agg(pl.col('item_idx')).to_dicts()}
    valid = {r['user_id']: r['item_idx'] for r in va.to_dicts()}
    test = {r['user_id']: r['item_idx'] for r in te.to_dicts()}
    return vocab, train, valid, test

def load_content(vocab):
    blocks = []
    for name in ('image', 'text'):
        vec = np.load(EMB_DIR/f'{name}_embeddings.npy', mmap_mode='r')
        meta = pl.read_parquet(EMB_DIR/f'{name}_embedding_metadata.parquet', columns=['item_id'])['item_id'].to_list()
        block = np.zeros((vocab.vocab_size, vec.shape[1]), dtype=np.float32)
        for row, item_id in enumerate(meta):
            if (idx := vocab.item2idx.get(item_id)) is not None: block[idx] = vec[row]
        blocks.append(block)
    return np.concatenate(blocks, axis=1)

vocab, train_seq, valid_targets, test_targets = load_sequences()
content_matrix = load_content(vocab)
print(f'Catalog={vocab.num_items:,}; content={content_matrix.shape}')

In [ ]:
class FrozenUserTower(nn.Module):
    def __init__(self, vocab_size, content_matrix, d_model=128, max_seq_len=20, n_heads=4, n_layers=2, d_ff=512, dropout=0.2):
        super().__init__(); self.d_model = d_model
        self.id_residual = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.register_buffer('content_matrix', torch.from_numpy(content_matrix).float())
        self.content_proj = nn.Linear(content_matrix.shape[1], d_model, bias=False)
        self.position_embedding = nn.Embedding(max_seq_len, d_model); self.embed_dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False); self.out_norm = nn.LayerNorm(d_model)
    def item_vectors(self):
        return self.id_residual.weight + self.content_proj(self.content_matrix)
    def encode_user(self, seq, item_table=None):
        table = self.item_vectors() if item_table is None else item_table
        b, length = seq.shape; positions = torch.arange(length, device=seq.device)[None].expand(b, -1)
        hidden = table[seq] * math.sqrt(self.d_model) + self.position_embedding(positions)
        mask = torch.triu(torch.ones(length, length, dtype=torch.bool, device=seq.device), diagonal=1)
        hidden = self.encoder(self.embed_dropout(hidden), mask=mask, src_key_padding_mask=seq.eq(PAD_IDX))
        hidden = self.out_norm(hidden); last = seq.ne(PAD_IDX).sum(1).clamp(min=1) - 1
        return hidden[torch.arange(b, device=seq.device), last]

tower = FrozenUserTower(vocab.vocab_size, content_matrix).to(device)
tower.load_state_dict(torch.load(USER_TOWER_DIR/'user_tower.pt', map_location=device)); tower.eval()
for p in tower.parameters(): p.requires_grad_(False)
print('✅ Frozen User Tower checkpoint loaded')

In [ ]:
@dataclass(frozen=True)
class Example:
    user_id: str
    context: np.ndarray
    target: int
    seen: frozenset

def examples_for(split):
    targets = valid_targets if split == 'valid' else test_targets; out = []
    for user_id, seq in train_seq.items():
        target = targets.get(user_id)
        if target is None: continue
        context_seq = seq if split == 'valid' else seq + ([valid_targets[user_id]] if user_id in valid_targets else [])
        seen = set(context_seq); seen.discard(target)
        out.append(Example(user_id, pad_right(context_seq), target, frozenset(seen)))
    return out

all_valid, test_examples = examples_for('valid'), examples_for('test')
split_rng = np.random.default_rng(SEED); order = split_rng.permutation(len(all_valid)); cut = int(.8 * len(order))
rerank_train = [all_valid[i] for i in order[:cut]]; rerank_valid = [all_valid[i] for i in order[cut:]]

counts = np.zeros(vocab.vocab_size, dtype=np.float32)
for seq in train_seq.values(): np.add.at(counts, np.asarray(seq, dtype=np.int64), 1)
pop_score = np.zeros(vocab.vocab_size, dtype=np.float32); log_pop = np.log1p(counts[1:vocab.num_items+1])
pop_score[1:vocab.num_items+1] = (log_pop - log_pop.mean()) / max(log_pop.std(), 1e-6)
pop_order = np.lexsort((np.arange(vocab.vocab_size), -counts)); pop_order = pop_order[(pop_order != PAD_IDX) & (pop_order != vocab.oov_idx)]
pop_rank = np.full(vocab.vocab_size, vocab.num_items, dtype=np.float32); pop_rank[pop_order] = np.arange(len(pop_order))
print(f'Rerank train/valid/test = {len(rerank_train):,}/{len(rerank_valid):,}/{len(test_examples):,}')

In [ ]:
@torch.no_grad()
def candidate_features(examples, force_target=False, batch_size=128):
    """Return one candidate-id array and numeric feature matrix per user."""
    table = tower.item_vectors(); norm_table = F.normalize(table, dim=-1)
    raw_content = F.normalize(tower.content_matrix, dim=-1); rows = []
    for start in range(0, len(examples), batch_size):
        batch = examples[start:start+batch_size]; context = torch.from_numpy(np.stack([x.context for x in batch])).to(device)
        users = tower.encode_user(context, table); scores = users @ table.T
        mask = context.ne(PAD_IDX); lengths = mask.sum(1, keepdim=True)
        positions = torch.arange(context.shape[1], device=device).float()[None]
        recency = (0.8 ** (lengths - 1 - positions).clamp(min=0)) * mask
        content_hist = raw_content[context]
        content_query = F.normalize((content_hist * recency.unsqueeze(-1)).sum(1), dim=-1)
        last_ids = context.gather(1, (lengths.long() - 1).clamp(min=0)).squeeze(1)
        content_scores = content_query @ raw_content.T
        last_scores = raw_content[last_ids] @ raw_content.T
        for matrix in (scores, content_scores, last_scores):
            matrix[:, PAD_IDX] = matrix[:, vocab.oov_idx] = float('-inf')
        for i, ex in enumerate(batch):
            if ex.seen:
                for matrix in (scores, content_scores, last_scores): matrix[i, list(ex.seen)] = float('-inf')
        top_ut = torch.topk(scores, UT_CANDIDATES, dim=1).indices
        top_content = torch.topk(content_scores, CONTENT_CANDIDATES, dim=1).indices
        top_last = torch.topk(last_scores, LAST_ITEM_CANDIDATES, dim=1).indices
        for i, ex in enumerate(batch):
            ut = top_ut[i].cpu().tolist(); content = top_content[i].cpu().tolist()
            last = top_last[i].cpu().tolist(); pop = []
            for item in pop_order:
                item = int(item)
                if item not in ex.seen:
                    pop.append(item)
                    if len(pop) == POP_CANDIDATES: break
            candidates = list(dict.fromkeys(ut + pop + content + last))
            if force_target and ex.target not in candidates: candidates.append(ex.target)
            ids = torch.tensor(candidates, device=device); raw = scores[i, ids]
            hist = context[i][context[i].ne(PAD_IDX)]; sims = norm_table[ids] @ norm_table[hist].T
            candidate_np = np.asarray(candidates, dtype=np.int64)
            def reciprocal_rank(source):
                ranks = {item: rank for rank, item in enumerate(source)}
                return torch.tensor([1.0 / (ranks[item] + 1) if item in ranks else 0.0 for item in candidates], device=device)
            rr_ut, rr_content, rr_last = reciprocal_rank(ut), reciprocal_rank(content), reciprocal_rank(last)
            rr_pop = torch.from_numpy(1 / (pop_rank[candidate_np] + 1)).to(device)
            features = torch.stack((raw, torch.from_numpy(pop_score[candidate_np]).to(device),
                rr_ut, rr_pop, content_scores[i, ids], last_scores[i, ids], rr_content, rr_last,
                sims[:, -1], sims.mean(1), sims.max(1).values,
                torch.full((len(ids),), float(len(hist)), device=device),
                rr_ut.gt(0).float(), rr_content.gt(0).float(), rr_last.gt(0).float()), dim=1)
            rows.append((candidate_np, features.cpu().numpy().astype(np.float32)))
    return rows

def listwise_arrays_from_examples(examples, negatives=LISTWISE_NEGATIVES, chunk_size=512):
    # Stream candidate chunks: retain one positive and a compact sampled list per user.
    rng = np.random.default_rng(SEED); pos, neg = [], []
    for start in range(0, len(examples), chunk_size):
        batch = examples[start:start+chunk_size]
        for (ids, features), ex in zip(candidate_features(batch, force_target=False), batch):
            where = np.flatnonzero(ids == ex.target)
            if not len(where): continue
            p = features[where[0]]; choices = np.delete(np.arange(len(ids)), where[0])
            n_hard = min(int(round(negatives * HARD_NEGATIVE_FRACTION)), len(choices))
            hard_score = features[choices][:, [2, 3, 6, 7]].max(axis=1)
            hard_order = np.argsort(-hard_score)
            hard_pool = choices[hard_order[:min(len(hard_order), max(256, n_hard))]]
            hard = rng.choice(hard_pool, size=n_hard, replace=len(hard_pool) < n_hard)
            remaining = negatives - n_hard
            random_part = rng.choice(choices, size=remaining, replace=len(choices) < remaining)
            pos.append(p); neg.append(features[np.concatenate((hard, random_part))])
        print(f'candidate train {min(start+chunk_size, len(examples)):,}/{len(examples):,}')
    return np.asarray(pos, dtype=np.float32), np.asarray(neg, dtype=np.float32)

print('Generating leakage-safe listwise samples from frozen tower...')
# Không force target: reranker chỉ học trên candidate mà retrieval thực sự sinh được.
x_pos, x_neg = listwise_arrays_from_examples(rerank_train)
valid_rows = candidate_features(rerank_valid, force_target=False)

In [ ]:
feature_mean = np.concatenate((x_pos, x_neg.reshape(-1, x_neg.shape[-1]))).mean(0)
feature_std = np.concatenate((x_pos, x_neg.reshape(-1, x_neg.shape[-1]))).std(0).clip(1e-6)
x_pos = (x_pos-feature_mean)/feature_std; x_neg = (x_neg-feature_mean)/feature_std

FEATURE_NAMES = ('tower_score', 'popularity_z', 'rr_tower', 'rr_popularity',
    'content_centroid_score', 'last_item_content_score', 'rr_content', 'rr_last_item',
    'tower_sim_last', 'tower_sim_mean', 'tower_sim_max', 'history_length',
    'from_tower', 'from_content', 'from_last_item')

class ResidualListwiseRanker(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(features, 64), nn.GELU(), nn.Dropout(.10),
            nn.Linear(64, 32), nn.GELU(), nn.Linear(32, 1))
        # Epoch 0 exactly reproduces User Tower ranking.
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, x, blend=1.0):
        return x[..., 0] + blend * self.net(x).squeeze(-1)

ranker = ResidualListwiseRanker(x_pos.shape[1]).to(device)
optimizer = torch.optim.AdamW(ranker.parameters(), lr=LR, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.from_numpy(x_pos), torch.from_numpy(x_neg)), batch_size=RERANK_BATCH_SIZE, shuffle=True)
print(f'Listwise train groups={len(x_pos):,}; candidates/group={x_neg.shape[1]+1}; features={x_pos.shape[1]}')

In [ ]:
KS = (10, 50, 100)

@torch.no_grad()
def evaluation_counts(rows, examples, blend):
    ranker.eval(); hits = {k: 0 for k in KS}; ndcg = {k: 0.0 for k in KS}; covered = 0
    for (ids, features), ex in zip(rows, examples):
        target_at = np.flatnonzero(ids == ex.target)
        if not len(target_at): continue
        covered += 1; x = torch.from_numpy((features-feature_mean)/feature_std).to(device)
        order = ranker(x, blend=blend).argsort(descending=True).cpu().numpy()
        rank = int(np.flatnonzero(order == target_at[0])[0])
        gain = 1.0 / math.log2(rank + 2)
        for k in KS:
            if rank < k: hits[k] += 1; ndcg[k] += gain
    return covered, hits, ndcg

def metrics_from_counts(n, covered, hits, ndcg):
    metrics = {'n_users': n, 'candidate_recall': covered/n}
    for k in KS:
        metrics[f'HitRate@{k}'] = hits[k]/n
        metrics[f'NDCG@{k}'] = ndcg[k]/n
        metrics[f'ConditionalHR@{k}'] = hits[k]/max(covered, 1)
    return metrics

def evaluate_reranker(rows, examples, blend):
    return metrics_from_counts(len(examples), *evaluation_counts(rows, examples, blend))

def selection_score(metrics):
    return metrics['NDCG@10'] + 0.5*metrics['NDCG@50'] + 0.25*metrics['NDCG@100']

baseline_metrics = evaluate_reranker(valid_rows, rerank_valid, blend=0.0)
best_metrics, best_blend, best_epoch = baseline_metrics, 0.0, 0
best_score, patience, history = selection_score(baseline_metrics), 0, []
np.savez(ARTIFACTS_DIR/'feature_scaler.npz', mean=feature_mean, std=feature_std)
torch.save({'model_state_dict': ranker.state_dict(), 'epoch': 0, 'blend': 0.0,
    'feature_names': FEATURE_NAMES, 'validation_metrics': baseline_metrics}, ARTIFACTS_DIR/'reranker.pt')
print('Validation retrieval baseline:', baseline_metrics)

for epoch in range(1, EPOCHS+1):
    ranker.train(); total = 0.
    for pos, neg in loader:
        pos, neg = pos.to(device), neg.to(device)
        group = torch.cat((pos[:, None, :], neg), dim=1)
        logits = ranker(group, blend=1.0)
        loss = F.cross_entropy(logits, torch.zeros(len(pos), dtype=torch.long, device=device))
        optimizer.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(ranker.parameters(), 5.0)
        optimizer.step(); total += loss.item()*len(pos)
    epoch_options = []
    for blend in BLEND_GRID:
        metrics = evaluate_reranker(valid_rows, rerank_valid, blend)
        eligible = metrics['HitRate@100'] >= baseline_metrics['HitRate@100'] - HR100_TOLERANCE
        epoch_options.append((eligible, selection_score(metrics), blend, metrics))
    eligible_options = [x for x in epoch_options if x[0]]
    chosen = max(eligible_options or epoch_options, key=lambda x: x[1])
    _, score, blend, metrics = chosen
    history.append({'epoch': epoch, 'loss': total/len(x_pos), 'blend': blend,
        'selection_score': score, 'metrics': metrics})
    print(f'Epoch {epoch:02d} loss={total/len(x_pos):.4f} blend={blend:.2f} valid={metrics}')
    if score > best_score + 1e-8:
        best_score, best_metrics, best_blend, best_epoch, patience = score, metrics, blend, epoch, 0
        torch.save({'model_state_dict': ranker.state_dict(), 'epoch': epoch, 'blend': blend,
            'feature_names': FEATURE_NAMES, 'validation_metrics': metrics}, ARTIFACTS_DIR/'reranker.pt')
    else: patience += 1
    if patience >= PATIENCE: break

checkpoint = torch.load(ARTIFACTS_DIR/'reranker.pt', map_location=device)
ranker.load_state_dict(checkpoint['model_state_dict']); best_blend = float(checkpoint['blend'])
run_config = {'seed': SEED, 'objective': 'residual_listwise_sampled_softmax',
    'listwise_negatives': LISTWISE_NEGATIVES, 'hard_negative_fraction': HARD_NEGATIVE_FRACTION,
    'candidate_budget': {'tower': UT_CANDIDATES, 'popularity': POP_CANDIDATES,
        'content_centroid': CONTENT_CANDIDATES, 'last_item': LAST_ITEM_CANDIDATES},
    'blend_grid': BLEND_GRID, 'hr100_tolerance': HR100_TOLERANCE,
    'best_epoch': int(checkpoint['epoch']), 'best_blend': best_blend}
(ARTIFACTS_DIR/'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
(ARTIFACTS_DIR/'training_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
(ARTIFACTS_DIR/'validation_metrics.json').write_text(json.dumps(checkpoint['validation_metrics'], indent=2), encoding='utf-8')

# Stream test candidates so a 16 GB machine never retains all ~2,000 x 21k rows.
test_covered, test_hits = 0, {k: 0 for k in KS}; test_ndcg = {k: 0.0 for k in KS}
TEST_CHUNK_SIZE = 512
for start in range(0, len(test_examples), TEST_CHUNK_SIZE):
    batch = test_examples[start:start+TEST_CHUNK_SIZE]
    covered, hits, ndcg = evaluation_counts(candidate_features(batch, force_target=False), batch, best_blend)
    test_covered += covered
    for k in KS: test_hits[k] += hits[k]; test_ndcg[k] += ndcg[k]
    print(f'test {min(start+TEST_CHUNK_SIZE, len(test_examples)):,}/{len(test_examples):,}')
test_metrics = metrics_from_counts(len(test_examples), test_covered, test_hits, test_ndcg)
(ARTIFACTS_DIR/'test_metrics.json').write_text(json.dumps(test_metrics, indent=2), encoding='utf-8')
print('TEST RESIDUAL LISTWISE RERANKER:', json.dumps(test_metrics, indent=2))